# Ancestree — Chaos Engineering Stress Suite

A deliberately hostile, break-it-on-purpose test pass against `ancestree`
(`LineageStore`). The goal is **not** the happy path: we try to crash the
process, corrupt the index, defeat deduplication, poison the chunk pool, and
drive every public method in illogical orders.

**What this library actually is** (so the chaos is aimed correctly): ancestree
is a *file-based data-lineage + experiment tracker*, not an Airflow-style
scheduler. The public surface is a single `LineageStore` class (no CLI); the
"DAG" is the **lineage** graph (multi-parent supported); "commands" are methods.
There is a background chunk-packer thread, an on-disk index (snapshot + journal)
that rebuilds from `meta.json`, content deduplication, and a self-contained HTML
explorer. Every chapter below targets one of those real seams.

Each probe is classified live:

| Verdict | Meaning |
|---|---|
| ✅ SURVIVED | handled correctly / failed safely as designed |
| 💥 BROKE | crashed, corrupted state, or produced an unrecoverable error |
| ⚠️ DEGRADED | did not crash but produced wrong / surprising / non-conformant output |
| 🔷 BY-DESIGN | a documented trade-off worth flagging to users |

Findings are collated into a table at the end. Everything runs against
throwaway stores in a temp dir; nothing touches your real data.

In [1]:
import json
import os
import random
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import warnings
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
from ancestree.vis import visualise_nodes

import ancestree
from ancestree import LineageStore

try:
    import psutil

    _PROC = psutil.Process()

    def rss_mb():
        return _PROC.memory_info().rss / 1e6
except Exception:

    def rss_mb():
        return float("nan")


print(
    "ancestree",
    ancestree.__version__,
    "| python",
    sys.version.split()[0],
    "| cpus",
    os.cpu_count(),
)

# ---- Tunable scale knobs (crank these up to push harder) --------------------
SCALE = dict(
    DAG_NODES=550,  # mega-DAG size (>500 as requested)
    FANOUT=100,  # 1 -> 100 fan-out
    FANIN=80,  # 80 -> 1 fan-in (join)
    DEEP=1100,  # deep chain (> default recursion limit of 1000)
    BIG_MB=32,  # "large" artifact size for chunk IO
    N_PROCS=6,  # concurrent OS processes
    PROC_NODES=25,  # nodes per process
    THREADS=8,  # threads for the shared-store storm
    MANY_PARAMS=5000,  # params logged onto a single node
)
print("SCALE:", SCALE)

WORK = Path(tempfile.mkdtemp(prefix="ancestree_chaos_"))
print("work dir:", WORK)


# ---- Chaos harness ----------------------------------------------------------
class Chaos:
    SURV, BROKE, DEGR, DESIGN, INFO = (
        "✅ SURVIVED",
        "💥 BROKE",
        "⚠️ DEGRADED",
        "🔷 BY-DESIGN",
        "ℹ️ INFO",
    )

    def __init__(self):
        self.rows = []

    def add(self, chapter, title, verdict, note="", severity=""):
        self.rows.append(
            dict(
                chapter=chapter,
                title=title,
                verdict=verdict,
                severity=severity,
                note=note,
            )
        )
        line = f"  {verdict}  {title}"
        if note:
            line += f"\n        └─ {note}"
        if severity:
            line += f"   [{severity}]"
        print(line)

    def counts(self):
        c = {}
        for r in self.rows:
            c[r["verdict"]] = c.get(r["verdict"], 0) + 1
        return c


R = Chaos()


def chapter(title):
    print("\n" + "█" * 78 + f"\n█  {title}\n" + "█" * 78)


def capture(fn, *a, **k):
    """Run fn; return (ok, result, exception). Never raises."""
    try:
        return True, fn(*a, **k), None
    except BaseException as e:  # noqa: BLE001 - we are *trying* to catch everything
        return False, None, e


def exc(e):
    return f"{type(e).__name__}: {e}".replace("\n", " ")[:160]


_store_seq = 0


def fresh(dedupe=False, chunk=False, rules=None, triggers=None):
    """A brand-new throwaway store under the work dir."""
    global _store_seq
    _store_seq += 1
    root = WORK / f"store_{_store_seq:03d}"
    return LineageStore(
        root=root, dedupe=dedupe, chunk=chunk, rules=rules, gen_triggers=triggers
    )


warnings.simplefilter("ignore")  # we assert on behaviour, not warning spam

ancestree 0.1.0 | python 3.12.12 | cpus 16
SCALE: {'DAG_NODES': 550, 'FANOUT': 100, 'FANIN': 80, 'DEEP': 1100, 'BIG_MB': 32, 'N_PROCS': 6, 'PROC_NODES': 25, 'THREADS': 8, 'MANY_PARAMS': 5000}
work dir: /var/folders/xf/n_m7ztrx4x577r3935n_1m1w0000gn/T/ancestree_chaos_t5i71_x3


## Chapter 1 — The Mega-Pipeline (500+ node randomized DAG)

Build a large lineage DAG mixing every awkward shape: a backbone chain, a wide
**1→100 fan-out**, a massive **80→1 fan-in (join)**, **diamonds**, and random
extra cross-edges — then hammer the topology-sensitive operations
(`get_lineage`, `prune`, the web graph). We're hunting for cycles, deadlocks,
recursion blow-ups, and lineage that doesn't come back topologically ordered.

In [2]:
chapter("CHAPTER 1 — MEGA-PIPELINE: 550-node randomized DAG")
rng = random.Random(1234)

store = fresh(chunk=False)
ids = []  # node_ids in creation order (a node's parents must precede it)
parents_of = {}  # id -> list of parent ids (ground truth we build)
t0 = time.time()


def mk(step, parents):
    with store.create_node(step_type=step, parent=parents or None) as n:
        n.add_meta("shape", step)
    # parents may be Node objects or node-id strings; record ground-truth ids.
    parents_of[n.node_id] = [getattr(p, "node_id", p) for p in (parents or [])]
    ids.append(n.node_id)
    return n


# 1) backbone chain of 50
prev = None
backbone = []
for i in range(50):
    prev = mk("chain", [prev] if prev else [])
    backbone.append(prev)

# 2) wide fan-out 1 -> FANOUT off the backbone tip
hub = backbone[-1]
fan = [mk("fanout", [hub]) for _ in range(SCALE["FANOUT"])]

# 3) massive fan-in: join FANIN of those into one node
join_inputs = fan[: SCALE["FANIN"]]
joined = mk("join", join_inputs)

# 4) a pile of diamonds: a -> {b,c} -> d
n_diamond = 0
while len(ids) < SCALE["DAG_NODES"] - 4:
    a = mk("d_top", [rng.choice(ids)])
    b = mk("d_left", [a])
    c = mk("d_right", [a])
    mk("d_join", [b, c])  # diamond closes here
    n_diamond += 1

# 5) a few random extra cross-edges (still a DAG: parents are older nodes)
for _ in range(20):
    child_pos = rng.randint(10, len(ids) - 1)
    # choose 1-3 strictly-older parents
    k = rng.randint(1, 3)
    older = [store.get_node(ids[rng.randint(0, child_pos - 1)]) for _ in range(k)]
    mk("cross", older)

build_s = time.time() - t0
N = len(ids)
print(
    f"\nbuilt {N} nodes ({n_diamond} diamonds) in {build_s:.1f}s "
    f"({build_s / N * 1000:.1f} ms/node)"
)


# -- Probe: no cycles, lineage is correct + topologically ordered -------------
def is_topo(order, truth):
    pos = {nid: i for i, nid in enumerate(order)}
    for nid in order:
        for p in truth.get(nid, []):
            if p in pos and pos[p] > pos[nid]:
                return False
    return True


sample = rng.sample(ids, 40)
ok_all, bad = True, None
for nid in sample:
    ok, lin, e = capture(store.get_lineage, nid)
    if not ok:
        ok_all = False
        bad = (nid, exc(e))
        break
    order = [n.node_id for n in lin]
    # every ancestor present, target last, topologically ordered
    if order[-1] != nid or not is_topo(order, parents_of):
        ok_all = False
        bad = (nid, "lineage not topo-ordered / target not last")
        break
if ok_all:
    R.add(
        "1-DAG",
        "get_lineage on 40 random nodes of a 550-node DAG",
        R.SURV,
        "all returned full ancestry, topologically ordered, no cycles",
    )
else:
    R.add("1-DAG", "get_lineage on the mega-DAG", R.BROKE, str(bad), "High")

# -- Probe: timing of queries at this scale -----------------------------------
t = time.time()
[store.find_node(shape="join") for _ in range(20)]
find_ms = (time.time() - t) / 20 * 1000
t = time.time()
store.get_lineage(joined)
lin_ms = (time.time() - t) * 1000
R.add(
    "1-DAG",
    f"query speed @ {N} nodes",
    R.INFO,
    f"find_node ≈ {find_ms:.1f} ms, get_lineage(join) ≈ {lin_ms:.1f} ms",
)

# -- Probe: orphan-only prune on a join (the tricky DAG case) ------------------
# Pruning ONE parent of the join must spare the join (other parents survive)
# and drop only the pruned id from its parent_id.
before = len(store.find_node())
pre_parents = len(store.get_node(joined.node_id).parent_id)
preview = store.prune(join_inputs[0], dry_run=True)
deleted = store.prune(join_inputs[0], dry_run=False)
survived = store.get_node(joined.node_id)
ok = (
    survived is not None
    and len(survived.parent_id) == pre_parents - 1
    and len(preview) == len(deleted)
)
R.add(
    "1-DAG",
    "orphan-only prune of one join-parent",
    R.SURV if ok else R.BROKE,
    f"join kept, parent_id {pre_parents}->{len(survived.parent_id) if survived else 'GONE'}; "
    f"deleted {len(deleted)} (preview matched={len(preview) == len(deleted)})",
    "" if ok else "High",
)

# -- Probe: web graph renders the whole DAG -----------------------------------
ok, p, e = capture(store.generate_web_graph)
if ok and p.exists():
    R.add(
        "1-DAG",
        "generate_web_graph over the whole DAG",
        R.SURV,
        f"{p.stat().st_size / 1e6:.2f} MB self-contained HTML",
    )
else:
    R.add("1-DAG", "generate_web_graph over the whole DAG", R.BROKE, exc(e), "High")


██████████████████████████████████████████████████████████████████████████████
█  CHAPTER 1 — MEGA-PIPELINE: 550-node randomized DAG
██████████████████████████████████████████████████████████████████████████████



built 567 nodes (99 diamonds) in 8.9s (15.6 ms/node)
  ✅ SURVIVED  get_lineage on 40 random nodes of a 550-node DAG
        └─ all returned full ancestry, topologically ordered, no cycles
  ℹ️ INFO  query speed @ 567 nodes
        └─ find_node ≈ 0.1 ms, get_lineage(join) ≈ 0.2 ms
  ✅ SURVIVED  orphan-only prune of one join-parent
        └─ join kept, parent_id 80->79; deleted 1 (preview matched=True)
  ✅ SURVIVED  generate_web_graph over the whole DAG
        └─ 1.57 MB self-contained HTML


### Deep chain — does `get_lineage`/`prune` survive past the recursion limit?
A linear chain longer than Python's default recursion limit (1000). `prune` and
`get_lineage` were both made iterative (known fixed findings); we re-verify at
scale and check the iterative claim holds.

In [3]:
deep = fresh(chunk=False)
prev = None
t0 = time.time()
for i in range(SCALE["DEEP"]):
    with deep.create_node(step_type="s", parent=prev) as n:
        n.add_meta("i", i)
    prev = n
print(f"built deep chain of {SCALE['DEEP']} in {time.time() - t0:.1f}s")

ok, lin, e = capture(deep.get_lineage, prev)
R.add(
    "1-DEEP",
    f"get_lineage on a {SCALE['DEEP']}-deep chain (> recursion limit)",
    R.SURV if ok and len(lin) == SCALE["DEEP"] else R.BROKE,
    f"returned {len(lin) if ok else '—'} nodes, oldest→newest" if ok else exc(e),
    "" if ok else "High",
)

ok, prev_dry, e = capture(deep.prune, deep.get_lineage(prev)[0], True)
R.add(
    "1-DEEP",
    f"prune preview of the {SCALE['DEEP']}-deep chain",
    R.SURV if ok else R.BROKE,
    f"would delete {len(prev_dry)}" if ok else exc(e),
    "" if ok else "High",
)

built deep chain of 1100 in 17.6s
  ✅ SURVIVED  get_lineage on a 1100-deep chain (> recursion limit)
        └─ returned 1100 nodes, oldest→newest
  ✅ SURVIVED  prune preview of the 1100-deep chain
        └─ would delete 1100


## Chapter 2 — IO & Data-Corruption Stressors

Drive the chunk store with large artifacts, then deliberately damage the
on-disk state the library promises to survive: truncated/garbage/missing
chunks, a corrupt `meta.json`, a corrupt index snapshot, a torn journal line,
and a leftover `meta.json.tmp` from a simulated crash mid-write.

In [4]:
chapter("CHAPTER 2 — IO & CORRUPTION")

# -- Large artifact + sub-file dedup -----------------------------------------
cs = fresh(chunk=True, dedupe=False)
# Truly random (incompressible, non-repeating) so the only sharing is between
# the two near-identical files — an honest sub-file-dedup demonstration.
big = os.urandom(SCALE["BIG_MB"] * 1024 * 1024)
with cs.create_node(step_type="big") as n1:
    (n1 / "a.bin").write_bytes(big)
# near-identical: flip a few bytes in the middle (CDC should share most chunks)
big2 = bytearray(big)
big2[len(big2) // 2] ^= 0xFF
with cs.create_node(step_type="big") as n2:
    (n2 / "b.bin").write_bytes(bytes(big2))
cs.flush()
logical = len(big) + len(big2)
pool = sum(f.stat().st_size for f in (cs.root / ".chunks").rglob("*") if f.is_file())
read_back = (cs.get_node(n1.node_id) / "a.bin").read_bytes()
ok = read_back == big
R.add(
    "2-IO",
    f"{SCALE['BIG_MB']}MB artifact pack + reassemble round-trip",
    R.SURV if ok else R.BROKE,
    f"bytes identical={ok}; pool {pool / 1e6:.1f}MB vs logical {logical / 1e6:.1f}MB "
    f"(near-identical files shared chunks)",
    "" if ok else "High",
)

# -- Chunk corruption: three distinct failure modes ---------------------------
cc = fresh(chunk=True)
with cc.create_node(step_type="m") as n:
    (n / "big.bin").write_bytes(b"A" * 200_000)
cc.flush()
chunks = [
    p
    for sh in (cc.root / ".chunks").iterdir()
    if sh.is_dir()
    for p in sh.iterdir()
    if p.is_file()
]
target = chunks[0]
original = target.read_bytes()


def read_artifact():
    cc.clear_cache()
    node = cc.get_node(n.node_id)
    return (node / "big.bin").read_bytes()


# mode A: garbage (not a valid zlib stream)
target.write_bytes(b"garbage not zlib")
ok, _, e = capture(read_artifact)
R.add(
    "2-IO",
    "read artifact with a GARBAGE chunk (invalid zlib)",
    R.DEGR,
    f"raises raw {type(e).__name__} from zlib, not the friendly integrity error",
    "Low",
)
# mode B: valid zlib, wrong content -> integrity check fires
target.write_bytes(zlib.compress(b"Z" * 64))
ok, _, e = capture(read_artifact)
friendly = e is not None and "integrity" in str(e).lower()
R.add(
    "2-IO",
    "read artifact with a WRONG-CONTENT chunk (valid zlib)",
    R.SURV if friendly else R.DEGR,
    f"{type(e).__name__}: {'clear integrity error' if friendly else str(e)[:60]}",
)
# mode C: missing chunk
target.unlink()
ok, _, e = capture(read_artifact)
R.add(
    "2-IO",
    "read artifact with a MISSING chunk",
    R.DEGR,
    f"raises bare {type(e).__name__}, not a clear 'chunk missing' error (known H8)",
    "Low",
)
target.parent.mkdir(parents=True, exist_ok=True)
target.write_bytes(original)

# -- Corrupt meta.json: does the advertised recovery survive it? --------------
cm = fresh(chunk=False)
kept = []
for i in range(4):
    with cm.create_node(step_type="m") as n:
        n.add_meta("i", i)
    kept.append(n.node_id)
victim = WORK / cm.root.name / kept[1] / "meta.json"
victim.write_text("{ this is not valid json")
# (a) index-backed query still answers from the snapshot
r2 = LineageStore(root=cm.root)
ok_q, res, _ = capture(lambda: len(r2.find_node()))
# (b) the ADVERTISED recovery path: rebuild_db_from_disk()
r3 = LineageStore(root=cm.root)
ok_rb, _, e_rb = capture(r3.rebuild_db_from_disk)
R.add(
    "2-IO",
    "one corrupt meta.json -> rebuild_db_from_disk() (the documented recovery)",
    R.BROKE if not ok_rb else R.SURV,
    f"rebuild raises {type(e_rb).__name__} instead of skipping the bad node "
    f"(confirms xfail 'test_rebuild_skips_corrupt_meta')"
    if not ok_rb
    else "skipped it",
    "Medium" if not ok_rb else "",
)
R.add(
    "2-IO",
    "corrupt meta.json of an already-indexed node (live query)",
    R.SURV if ok_q else R.BROKE,
    f"index still answers find_node()={res}; only direct get_node() of the bad "
    f"node returns None",
)

# -- Corrupt index snapshot ---------------------------------------------------
ci = fresh(chunk=False)
for i in range(3):
    with ci.create_node(step_type="m") as n:
        n.add_meta("i", i)
(ci.root / ".index.json").write_text("{ broken snapshot")
r4 = LineageStore(root=ci.root)
ok_load, _, e_load = capture(lambda: r4.find_node())
heals = ok_load
r5 = LineageStore(root=ci.root)
ok_rebuild, _, _ = capture(r5.rebuild_db_from_disk)
ok_after = ok_rebuild and len(r5.find_node()) == 3
R.add(
    "2-IO",
    "corrupt .index.json snapshot",
    R.SURV if (not heals and ok_after) else R.DEGR,
    "query raises a clear RuntimeError telling you to rebuild; rebuild fully "
    "recovers — but it does NOT self-heal on load (known xfail)",
)

# -- Torn final journal line (concurrent-append crash) ------------------------
tj = fresh(chunk=False)
for i in range(3):
    with tj.create_node(step_type="m") as n:
        n.add_meta("i", i)
log = tj.root / ".index.log"
if log.exists():
    with log.open("a") as f:
        f.write('{"id":"deadbeef","meta":{"step_t')  # torn, no newline
r6 = LineageStore(root=tj.root)
ok, res, e = capture(lambda: len(r6.find_node()))
R.add(
    "2-IO",
    "torn final line in .index.log (simulated crash mid-append)",
    R.SURV if ok and res == 3 else R.BROKE,
    f"torn line skipped, {res} nodes recovered from disk" if ok else exc(e),
)

# -- Leftover meta.json.tmp -> phantom artifact -------------------------------
pt = fresh(chunk=False)
with pt.create_node(step_type="m") as n:
    (n / "real.csv").write_text("data")
(WORK / pt.root.name / n.node_id / "meta.json.tmp").write_text("{half-written")
names = [p.name for p in pt.get_node(n.node_id).artifacts()]
phantom = "meta.json.tmp" in names
R.add(
    "2-IO",
    "leftover meta.json.tmp after a crash mid-write",
    R.DEGR if phantom else R.SURV,
    f"artifacts() lists {names} — the temp file shows up as a phantom artifact "
    f"(also pollutes content_hash & web graph; never reclaimed by the packer)"
    if phantom
    else "temp file correctly ignored",
    "Low" if phantom else "",
)


██████████████████████████████████████████████████████████████████████████████
█  CHAPTER 2 — IO & CORRUPTION
██████████████████████████████████████████████████████████████████████████████


  ✅ SURVIVED  32MB artifact pack + reassemble round-trip
        └─ bytes identical=True; pool 33.6MB vs logical 67.1MB (near-identical files shared chunks)
  ⚠️ DEGRADED  read artifact with a GARBAGE chunk (invalid zlib)
        └─ raises raw error from zlib, not the friendly integrity error   [Low]
  ✅ SURVIVED  read artifact with a WRONG-CONTENT chunk (valid zlib)
        └─ RuntimeError: clear integrity error
  ⚠️ DEGRADED  read artifact with a MISSING chunk
        └─ raises bare FileNotFoundError, not a clear 'chunk missing' error (known H8)   [Low]
  💥 BROKE  one corrupt meta.json -> rebuild_db_from_disk() (the documented recovery)
        └─ rebuild raises JSONDecodeError instead of skipping the bad node (confirms xfail 'test_rebuild_skips_corrupt_meta')   [Medium]
  ✅ SURVIVED  corrupt meta.json of an already-indexed node (live query)
        └─ index still answers find_node()=4; only direct get_node() of the bad node returns None
  ✅ SURVIVED  corrupt .index.json snapshot
   

  ✅ SURVIVED  torn final line in .index.log (simulated crash mid-append)
        └─ torn line skipped, 3 nodes recovered from disk
  ⚠️ DEGRADED  leftover meta.json.tmp after a crash mid-write
        └─ artifacts() lists ['meta.json.tmp', 'real.csv'] — the temp file shows up as a phantom artifact (also pollutes content_hash & web graph; never reclaimed by the packer)   [Low]


## Chapter 3 — Weird & Malicious Values

Bizarre types and payloads through `add_meta` and `create_node`: NaN/Inf,
empty/NaN DataFrames, pathologically deep and self-referential dicts, chaotic
encodings (unicode, emoji, null bytes), thousands of params, ultra-long keys,
and the system-key hijacks the reserved-key guard doesn't cover.

In [5]:
chapter("CHAPTER 3 — WEIRD & MALICIOUS VALUES")

# -- NaN / Inf : the big one --------------------------------------------------
nans = fresh(chunk=False)
with nans.create_node(step_type="m") as n:
    n.add_meta("score", float("nan"))
    n.add_meta("ratio", float("inf"))
    n.add_meta("negratio", float("-inf"))
raw = (WORK / nans.root.name / n.node_id / "meta.json").read_text()


# (a) on-disk meta.json is non-conformant JSON
def strict_load(s):
    return json.loads(
        s, parse_constant=lambda c: (_ for _ in ()).throw(ValueError(f"bad token {c}"))
    )


ok_strict, _, e_strict = capture(strict_load, raw)
# (b) the index journal too
logtxt = (
    (nans.root / ".index.log").read_text()
    if (nans.root / ".index.log").exists()
    else ""
)
# (c) the web graph's embedded JSON is non-conformant (breaks JS JSON.parse)
data = visualise_nodes(nans)
ok_web, _, _ = capture(lambda: json.dumps(data, allow_nan=False))
# (d) NaN is unsearchable (nan != nan)
found = nans.find_node(score=float("nan"))
R.add(
    "3-VAL",
    "NaN / Inf in metadata -> non-conformant JSON on disk",
    R.DEGR,
    f"meta.json contains literal NaN/Infinity tokens; strict JSON.parse "
    f"REJECTS it ({'rejected' if not ok_strict else 'ok?!'}); also written to "
    f".index.log ({'NaN' in logtxt})",
    "Medium",
)
R.add(
    "3-VAL",
    "NaN / Inf breaks the shareable web graph",
    R.DEGR,
    "generate_web_graph embeds NaN/Infinity; a browser's JSON.parse "
    "throws, so the whole interactive report fails to load",
    "Medium",
)
R.add(
    "3-VAL",
    "NaN metadata is unsearchable",
    R.DEGR,
    f"find_node(score=nan) -> {len(found)} matches (nan != nan, so a NaN "
    f"value can never be matched by equality)",
    "Low",
)

# -- HTML/JS injection into the shareable web graph (known H4) ----------------
xss = fresh(chunk=False)
with xss.create_node(step_type="m") as n:
    n.add_meta("label", "</script><img src=x onerror=alert(1)>")
html = xss.generate_web_graph().read_text()
injected = "</script><img src=x onerror=alert(1)>" in html
R.add(
    "3-VAL",
    "HTML/JS injection via metadata into the web graph",
    R.DEGR if injected else R.SURV,
    "metadata is embedded in the <script> block unescaped, so a </script> payload "
    "breaks out and the markup/JS runs — stored XSS in a file the docs tell you to "
    "email/commit/share (known H4)"
    if injected
    else "payload escaped",
    "Medium" if injected else "",
)

# -- Empty & NaN DataFrames as tables ----------------------------------------
dfs = fresh(chunk=False)


def _add_table(store, key, df):
    with store.create_node(step_type="m") as n:
        n.add_meta(key, df, data_type="table")


ok1, _, e1 = capture(_add_table, dfs, "empty", pd.DataFrame())
ok2, _, e2 = capture(
    _add_table, dfs, "nan_table", pd.DataFrame({"a": [1.0, np.nan], "b": [np.inf, 2.0]})
)
R.add(
    "3-VAL",
    "empty DataFrame as a table",
    R.SURV if ok1 else R.BROKE,
    "stored with empty columns/rows" if ok1 else exc(e1),
)
R.add(
    "3-VAL",
    "DataFrame containing NaN/Inf as a table",
    R.SURV if ok2 else R.BROKE,
    "stored — but inherits the NaN-in-JSON problem above (the table cells become "
    "NaN tokens in the web graph)"
    if ok2
    else exc(e2),
    "" if ok2 else "Medium",
)


# -- Deeply nested dict: find the depth ceiling -------------------------------
def nested(depth):
    d = cur = {}
    for _ in range(depth):
        cur["n"] = {}
        cur = cur["n"]
    return d


ceiling = None
for depth in (100, 500, 900, 1500, 3000):
    s = fresh(chunk=False)

    def _try(depth=depth, s=s):
        with s.create_node(step_type="m") as nn:
            nn.add_meta("deep", nested(depth))

    ok, _, e = capture(_try)
    if not ok:
        ceiling = (depth, type(e).__name__)
        break
R.add(
    "3-VAL",
    "deeply nested dict in metadata",
    R.DEGR if ceiling else R.SURV,
    f"add_meta raises {ceiling[1]} once nesting reaches ~{ceiling[0]} levels "
    f"(uncaught; surfaces inside the user's with-block)"
    if ceiling
    else "survived to 3000 deep",
    "Low" if ceiling else "",
)

# -- Circular reference -------------------------------------------------------
circ = fresh(chunk=False)
c = {}
c["self"] = c


def _try_circ():
    with circ.create_node(step_type="m") as nn:
        nn.add_meta("c", c)


ok, _, e = capture(_try_circ)
R.add(
    "3-VAL",
    "self-referential (circular) dict in metadata",
    R.DEGR if not ok else R.BROKE,
    f"raises {type(e).__name__} (no infinite loop, but not a clear 'circular "
    f"reference' message)"
    if not ok
    else "silently accepted (!)",
    "Low",
)

# -- Chaotic encodings: unicode / emoji / null bytes --------------------------
enc = fresh(chunk=False)
payloads = {
    "emoji_🔥_key": "value 🎉🤖",
    "ключ": "значение",
    "null\x00byte": "val\x00ue",
    "newline\nkey": "tab\tvalue",
    "x" * 1_000_000: "ultra-long key (1MB)",  # ultra-long metric NAME
}
enc_ok = True
enc_note = ""
try:
    with enc.create_node(step_type="m") as n:
        for k, v in payloads.items():
            n.add_meta(k, v)
    rl = enc.get_node(n.node_id)
    for k, v in payloads.items():
        if rl.metadata[k]["value"] != v:
            enc_ok = False
            enc_note = f"key {k[:20]!r} did not round-trip"
except Exception as e:
    enc_ok = False
    enc_note = exc(e)
R.add(
    "3-VAL",
    "unicode / emoji / null-byte / 1MB-long keys+values",
    R.SURV if enc_ok else R.DEGR,
    "all round-tripped byte-exact (incl. null bytes & a 1MB key)"
    if enc_ok
    else enc_note,
)


# null byte in STEP_TYPE specifically (used as a category key) -> must be rejected
def _bad_step(store, st):
    with store.create_node(step_type=st):
        pass


ok, _, e = capture(_bad_step, enc, "bad\x00type")
R.add(
    "3-VAL",
    "null byte in step_type (a category key)",
    R.SURV if (not ok and isinstance(e, ValueError)) else R.BROKE,
    "rejected at create_node (not printable)" if not ok else "accepted (!)",
)

# -- Thousands of params on one node -----------------------------------------
many = fresh(chunk=False)
t0 = time.time()
with many.create_node(step_type="m") as n:
    for i in range(SCALE["MANY_PARAMS"]):
        n.add_meta(f"param_{i}", i, group="sweep")
dt = time.time() - t0
rl = many.get_node(n.node_id)
idx_entry = many.database.cache[n.node_id]
R.add(
    "3-VAL",
    f"{SCALE['MANY_PARAMS']} params logged onto ONE node",
    R.SURV if len(rl.metadata) >= SCALE["MANY_PARAMS"] else R.BROKE,
    f"persisted in {dt:.2f}s; index entry holds {len(idx_entry)} searchable keys "
    f"(every key is indexed in-memory AND appended to .index.log)",
)

# -- Duplicate key, different types (overwrite semantics) ---------------------
dup = fresh(chunk=False)
with dup.create_node(step_type="m") as n:
    n.add_meta("k", 1)
    n.add_meta("k", "now a string")
    n.add_meta("k", [1, 2, 3])
final = dup.get_node(n.node_id).metadata["k"]["value"]
R.add(
    "3-VAL",
    "same key logged 3× with different types",
    R.SURV if final == [1, 2, 3] else R.BROKE,
    f"last-write-wins -> {final!r} (documented overwrite)",
)

# -- node_id hijack: the reserved-key guard gap -------------------------------
hj = fresh(chunk=False)
with hj.create_node(step_type="m") as n:
    (n / "x.txt").write_text("real")
    n.add_meta("node_id", "HACKED")  # NOT a reserved key -> allowed
    n.add_meta("user", "spoofed-user")  # provenance also unguarded
real_dir = n.node_id
entry = hj.database.cache[real_dir]
back = hj.find_node(step_type="m")[0]
lineage_ids = [x.node_id for x in hj.get_lineage(real_dir)]
ok_prune, _, e_prune = capture(hj.prune, back, False)
R.add(
    "3-VAL",
    "hijack node_id via add_meta (reserved-key guard gap)",
    R.BROKE,
    f"index entry node_id='{entry.get('node_id')}' but dir is '{real_dir}'; "
    f"find_node reports id='{back.node_id}'; get_lineage reports {lineage_ids}; "
    f"prune() then raises {type(e_prune).__name__ if e_prune else 'nothing'} "
    f"-> the node becomes un-prunable",
    "Medium",
)
R.add(
    "3-VAL",
    "provenance keys (user/git_commit/...) are unguarded too",
    R.DEGR,
    "add_meta('user', ...) silently overwrites captured provenance — the "
    "reproducibility record can be spoofed",
    "Low",
)

# -- Conflicting config on reopen / schema not enforced -----------------------
cfg_root = WORK / "conflict_store"
s1 = LineageStore(root=cfg_root, rules={"clean": ["ingest"]}, gen_triggers=["ingest"])
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    s2 = LineageStore(
        root=cfg_root, rules={"clean": ["TOTALLY_DIFFERENT"]}, gen_triggers=["other"]
    )
R.add(
    "3-VAL",
    "reopen a store with CONFLICTING rules",
    R.SURV if s2.rules == {"clean": ["ingest"]} else R.BROKE,
    f"stored rules win, supplied rules ignored with a UserWarning "
    f"({len(w)} warning(s)) — config is immutable by design",
)
# schema between node boundaries is NOT validated: a 'clean' node can write anything
sc = fresh(rules={"clean": ["ingest"]})
with sc.create_node(step_type="ingest") as a:
    (a / "expected.csv").write_text("x")
with sc.create_node(step_type="clean", parent=a) as b:
    (b / "WRONG_FORMAT.bin").write_bytes(b"\x00")
R.add(
    "3-VAL",
    "mismatched data schema across a node boundary",
    R.DESIGN,
    "rules constrain step_type transitions only, never artifact "
    "shape/schema — a 'clean' step can emit anything; not validated",
)


██████████████████████████████████████████████████████████████████████████████
█  CHAPTER 3 — WEIRD & MALICIOUS VALUES
██████████████████████████████████████████████████████████████████████████████
  ⚠️ DEGRADED  NaN / Inf in metadata -> non-conformant JSON on disk
        └─ meta.json contains literal NaN/Infinity tokens; strict JSON.parse REJECTS it (rejected); also written to .index.log (True)   [Medium]
  ⚠️ DEGRADED  NaN / Inf breaks the shareable web graph
        └─ generate_web_graph embeds NaN/Infinity; a browser's JSON.parse throws, so the whole interactive report fails to load   [Medium]
  ⚠️ DEGRADED  NaN metadata is unsearchable
        └─ find_node(score=nan) -> 0 matches (nan != nan, so a NaN value can never be matched by equality)   [Low]
  ⚠️ DEGRADED  HTML/JS injection via metadata into the web graph
        └─ metadata is embedded in the <script> block unescaped, so a </script> payload breaks out and the markup/JS runs — stored XSS in a file the docs tell you to ema

  ⚠️ DEGRADED  deeply nested dict in metadata
        └─ add_meta raises RecursionError once nesting reaches ~1500 levels (uncaught; surfaces inside the user's with-block)   [Low]


  ⚠️ DEGRADED  self-referential (circular) dict in metadata
        └─ raises RecursionError (no infinite loop, but not a clear 'circular reference' message)   [Low]
  ✅ SURVIVED  unicode / emoji / null-byte / 1MB-long keys+values
        └─ all round-tripped byte-exact (incl. null bytes & a 1MB key)
  ✅ SURVIVED  null byte in step_type (a category key)
        └─ rejected at create_node (not printable)
  ✅ SURVIVED  5000 params logged onto ONE node
        └─ persisted in 0.05s; index entry holds 5008 searchable keys (every key is indexed in-memory AND appended to .index.log)
  ✅ SURVIVED  same key logged 3× with different types
        └─ last-write-wins -> [1, 2, 3] (documented overwrite)
  💥 BROKE  hijack node_id via add_meta (reserved-key guard gap)
        └─ index entry node_id='HACKED' but dir is 'afb8eb85'; find_node reports id='HACKED'; get_lineage reports ['HACKED']; prune() then raises KeyError -> the node becomes un-prunable   [Medium]
  ⚠️ DEGRADED  provenance keys (user/

## Chapter 4 — Concurrency & Race Conditions

The library documents itself as **process-safe but NOT thread-safe**. We
pressure-test both claims: many OS processes writing one store, a barrier-synced
dedupe race, a write/`gc()` collision, and threads sharing one store (plus a
direct hammer of the index dict the docs warn about).

In [6]:
chapter("CHAPTER 4 — CONCURRENCY & RACES")

# A worker script run as real, separate OS processes (the honest multi-process
# test — no pickling of notebook closures, true interpreter isolation).
WORKER = WORK / "chaos_workers.py"
WORKER.write_text(r"""
import sys, json, time
from pathlib import Path
from ancestree import LineageStore

def barrier(bdir, idx, nprocs, tag):
    bdir = Path(bdir); bdir.mkdir(parents=True, exist_ok=True)
    (bdir / f"{tag}_{idx}.ready").write_text("1")
    deadline = time.time() + 30
    while time.time() < deadline:
        if len(list(bdir.glob(f"{tag}_*.ready"))) >= nprocs: return
        time.sleep(0.001)

def main():
    mode, root, idx, nprocs, nitems, bdir, out = (
        sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4]),
        int(sys.argv[5]), sys.argv[6], sys.argv[7])
    res = {"idx": idx, "mode": mode, "created": [], "errors": []}
    try:
        store = LineageStore(root=root, dedupe=True, chunk=True)
        if mode == "distinct":
            for i in range(nitems):
                with store.create_node(step_type="run") as n:
                    (n / "data.bin").write_bytes(f"proc{idx}-item{i}".encode() + b"\x00" * 2000)
                    n.add_meta("proc", idx); n.add_meta("i", i)
                res["created"].append(n.node_id)
            store.flush()
        elif mode == "same":
            barrier(bdir, idx, nprocs, "same")
            with store.create_node(step_type="dup") as n:
                (n / "data.bin").write_bytes(b"IDENTICAL-PAYLOAD" + b"\x00" * 8192)
                n.add_meta("shared", "value")
            res["created"].append(n.node_id)
            store.flush()
        elif mode == "writegc":
            barrier(bdir, idx, nprocs, "writegc")
            for i in range(nitems):
                with store.create_node(step_type="run") as n:
                    (n / "d.bin").write_bytes((f"p{idx}i{i}".encode()) * 500)
                res["created"].append(n.node_id)
                if idx == 0 and i % 3 == 0:
                    store.gc()          # collector races the writers
            store.flush()
            if idx == 0: store.gc()
    except BaseException as e:
        res["errors"].append(f"{type(e).__name__}: {e}")
    Path(out).write_text(json.dumps(res))

if __name__ == "__main__":
    main()
""")


def run_procs(mode, root, nprocs, nitems):
    bdir = WORK / f"barrier_{mode}_{int(time.time() * 1000)}"
    outs = [WORK / f"out_{mode}_{i}.json" for i in range(nprocs)]
    procs = [
        subprocess.Popen(
            [
                sys.executable,
                str(WORKER),
                mode,
                str(root),
                str(i),
                str(nprocs),
                str(nitems),
                str(bdir),
                str(outs[i]),
            ]
        )
        for i in range(nprocs)
    ]
    for p in procs:
        p.wait(timeout=120)
    results = [json.loads(o.read_text()) for o in outs if o.exists()]
    return results, [p.returncode for p in procs]


# -- 4a: many processes, distinct content -> no data loss ---------------------
root_a = WORK / "mp_distinct"
results, codes = run_procs("distinct", root_a, SCALE["N_PROCS"], SCALE["PROC_NODES"])
expected = SCALE["N_PROCS"] * SCALE["PROC_NODES"]
worker_errs = [e for r in results for e in r["errors"]]
final = LineageStore(root=root_a)
on_disk = len(list(root_a.glob("*/meta.json")))
indexed = len(final.find_node(step_type="run"))
# every artifact readable?
unreadable = 0
for nid in [c for r in results for c in r["created"]]:
    node = final.get_node(nid)
    try:
        _ = (node / "data.bin").read_bytes()
    except Exception:
        unreadable += 1
ok = (
    not worker_errs
    and indexed == expected == on_disk
    and unreadable == 0
    and all(c == 0 for c in codes)
)
R.add(
    "4-CONC",
    f"{SCALE['N_PROCS']} processes × {SCALE['PROC_NODES']} distinct nodes",
    R.SURV if ok else R.BROKE,
    f"expected {expected}: on_disk={on_disk}, indexed={indexed}, unreadable={unreadable}, "
    f"worker_errors={len(worker_errs)} — process-safety claim holds"
    if ok
    else f"on_disk={on_disk} indexed={indexed} unreadable={unreadable} errs={worker_errs[:2]}",
    "" if ok else "High",
)

# -- 4b: barrier-synced dedupe race ------------------------------------------
root_b = WORK / "mp_dedupe_race"
results, codes = run_procs("same", root_b, SCALE["N_PROCS"], 1)
final = LineageStore(root=root_b)
distinct = len({d.name for d in root_b.glob("*") if (d / "meta.json").exists()})
R.add(
    "4-CONC",
    f"{SCALE['N_PROCS']} processes create IDENTICAL content at a barrier",
    R.DESIGN if distinct >= 1 else R.BROKE,
    f"{distinct} distinct node(s) persisted for identical content "
    f"(dedupe is best-effort across processes: a tight race can leave up to "
    f"{SCALE['N_PROCS']} duplicates; never corruption)",
)

# -- 4c: writers racing a concurrent gc() ------------------------------------
root_c = WORK / "mp_writegc"
results, codes = run_procs("writegc", root_c, SCALE["N_PROCS"], SCALE["PROC_NODES"])
worker_errs = [e for r in results for e in r["errors"]]
final = LineageStore(root=root_c)
unreadable = 0
allids = [c for r in results for c in r["created"]]
for nid in allids:
    node = final.get_node(nid)
    if node is None:
        unreadable += 1
        continue
    try:
        _ = (node / "d.bin").read_bytes()
    except Exception:
        unreadable += 1
ok = not worker_errs and unreadable == 0 and all(c == 0 for c in codes)
R.add(
    "4-CONC",
    "writers racing a concurrent gc() (chunk reclamation)",
    R.SURV if ok else R.BROKE,
    f"{len(allids)} nodes, {unreadable} unreadable, {len(worker_errs)} worker errors "
    f"— the 60s grace + .gc.lock held"
    if ok
    else f"unreadable={unreadable}, errs={worker_errs[:2]} — gc reaped a live chunk!",
    "" if ok else "High",
)

# -- 4d: threads sharing ONE store (undocumented) -----------------------------
# (d1) the public API under an 8-thread storm
shared = fresh(chunk=False)
for i in range(60):
    with shared.create_node(step_type="m") as n:
        n.add_meta("i", i)
api_errs = []
stop = threading.Event()


def _w():
    i = 0
    while not stop.is_set():
        try:
            with shared.create_node(step_type="m") as n:
                n.add_meta("w", i)
                i += 1
        except Exception as e:
            api_errs.append(exc(e))
            return


def _r():
    while not stop.is_set():
        try:
            shared.find_node(step_type="m")
            shared.get_child_nodes("nope")
        except Exception as e:
            api_errs.append(exc(e))
            return


ts = [threading.Thread(target=_w) for _ in range(SCALE["THREADS"] // 2)] + [
    threading.Thread(target=_r) for _ in range(SCALE["THREADS"] // 2)
]
[t.start() for t in ts]
time.sleep(2.0)
stop.set()
[t.join(timeout=5) for t in ts]
reload = LineageStore(root=shared.root)
disk_n = len(list(shared.root.glob("*/meta.json")))
idx_n = len(reload.find_node(step_type="m"))
R.add(
    "4-CONC",
    f"{SCALE['THREADS']} threads sharing ONE store via the public API",
    R.INFO,
    f"{len(api_errs)} error(s) this run; {disk_n} on disk == {idx_n} indexed (no data "
    f"lost). The public API mostly absorbs the race (every read reloads from disk; "
    f"~15ms/node git cost dwarfs the window) but can surface a stray error — sharing "
    f"one store across threads is explicitly unsupported",
)

# (d2) hammer the index dict directly, as the docs warn against
db = fresh(chunk=False).database
for i in range(300):
    db.cache[f"s{i}"] = {
        "step_type": "m",
        "timestamp": "2020-01-01T00:00:00+00:00",
        "parent_id": [],
    }
raw_errs = []
stop = threading.Event()


def _mut():
    i = 0
    while not stop.is_set():
        try:
            db.cache[f"x{threading.get_ident()}_{i}"] = {
                "step_type": "m",
                "parent_id": [],
            }
            if i % 4 == 0 and db.cache:
                db.cache.pop(next(iter(db.cache)), None)
            i += 1
        except Exception as e:
            raw_errs.append(exc(e))
            return


def _it():
    while not stop.is_set():
        try:
            [k for k, m in db.cache.items() if m.get("step_type") == "m"]
        except Exception as e:
            raw_errs.append(exc(e))
            return


ts = [threading.Thread(target=_mut) for _ in range(4)] + [
    threading.Thread(target=_it) for _ in range(4)
]
[t.start() for t in ts]
time.sleep(1.5)
stop.set()
[t.join(timeout=5) for t in ts]
crashed = any("changed size during iteration" in e for e in raw_errs)
R.add(
    "4-CONC",
    "direct concurrent access to the index dict (what the docs forbid)",
    R.INFO,
    f"{len(raw_errs)} error(s); '{'dictionary changed size during iteration' if crashed else 'no crash this run'}'"
    f" — confirms the documented 'not thread-safe' limitation is real at the data-structure "
    f"layer, even though the public API survived above",
)


██████████████████████████████████████████████████████████████████████████████
█  CHAPTER 4 — CONCURRENCY & RACES
██████████████████████████████████████████████████████████████████████████████


  ✅ SURVIVED  6 processes × 25 distinct nodes
        └─ expected 150: on_disk=150, indexed=150, unreadable=0, worker_errors=0 — process-safety claim holds
  🔷 BY-DESIGN  6 processes create IDENTICAL content at a barrier
        └─ 1 distinct node(s) persisted for identical content (dedupe is best-effort across processes: a tight race can leave up to 6 duplicates; never corruption)


  ✅ SURVIVED  writers racing a concurrent gc() (chunk reclamation)
        └─ 150 nodes, 0 unreadable, 0 worker errors — the 60s grace + .gc.lock held


  ℹ️ INFO  8 threads sharing ONE store via the public API
        └─ 0 error(s) this run; 280 on disk == 280 indexed (no data lost). The public API mostly absorbs the race (every read reloads from disk; ~15ms/node git cost dwarfs the window) but can surface a stray error — sharing one store across threads is explicitly unsupported


  ℹ️ INFO  direct concurrent access to the index dict (what the docs forbid)
        └─ 4 error(s); 'dictionary changed size during iteration' — confirms the documented 'not thread-safe' limitation is real at the data-structure layer, even though the public API survived above


## Chapter 5 — All-Methods Assault & Fuzzing

Drive **every public method** in illogical orders (resume before create, query
a deleted run, prune the root, operate on a vanished store dir, double-close),
then a randomized fuzzer that throws garbage args at every method and flags any
*undocumented* exception type.

In [7]:
chapter("CHAPTER 5 — ALL-METHODS ASSAULT & FUZZING")

s = fresh(chunk=True, dedupe=True, rules={"clean": ["ingest"]}, triggers=["ingest"])
with s.create_node(step_type="ingest") as ing:
    (ing / "d.csv").write_text("x")
with s.create_node(step_type="clean", parent=ing) as cl:
    cl.add_meta("a", 1)


# All helpers defined up front (capture() runs the body immediately).
def _illegal():
    with s.create_node(step_type="clean", parent=None):
        pass  # clean needs an ingest parent


def _foreign():
    with s.create_node(step_type="clean", parent="not-a-real-id"):
        pass


def _prune_root():
    fake = s.get_node(ing.node_id)
    fake.path = s.root
    s._prune(fake, dry_run=False)


def _wipe_and_regen():
    for n in s.find_node():
        if s.get_node(n.node_id):
            capture(s.prune, n.node_id, False)
    return s.generate_web_graph()


def _recreate(store):
    with store.create_node(step_type="m") as n:
        n.add_meta("x", 1)


def expect(desc, fn, acceptable, severity=""):
    """Run fn; OK if it returns or raises one of the `acceptable` exception types."""
    ok, res, e = capture(fn)
    if ok:
        R.add("5-API", desc, R.SURV, f"returned {str(res)[:50]!r}")
    elif isinstance(e, acceptable):
        R.add("5-API", desc, R.SURV, f"failed safely: {type(e).__name__}")
    else:
        R.add("5-API", desc, R.BROKE, exc(e), severity or "Medium")


expect(
    "resume before anything exists: get_most_recent_node on empty store",
    lambda: fresh().get_most_recent_node(step_type="clean"),
    (type(None),),
)
expect("prune a never-created id", lambda: s.prune("deadbeef"), (type(None), list))
expect("get_node on garbage id", lambda: s.get_node("deadbeef"), (type(None),))
expect("create with an illegal transition (rules violation)", _illegal, (ValueError,))
expect("create with a foreign/unknown parent id", _foreign, (ValueError,))
expect("prune the store ROOT directory", _prune_root, (PermissionError, ValueError))
expect(
    "from_parent on a root node (no parents)",
    lambda: s.from_parent(ing, "*.csv"),
    (list,),
)
expect(
    "double context-exit / flush idempotency",
    lambda: (s.flush(), s.flush(), s.clear_cache())[-1],
    (type(None),),
)
expect(
    "prune EVERYTHING then regenerate the web graph",
    _wipe_and_regen,
    (type(None), Path),
)

# get_lineage / find_in_lineage on an unknown id: raise KeyError where siblings return None/[]
ok_gl, _, e_gl = capture(s.get_lineage, "deadbeef")
ok_fl, _, e_fl = capture(s.find_in_lineage, "deadbeef")
R.add(
    "5-API",
    "get_lineage / find_in_lineage on an unknown id",
    R.DEGR,
    f"both raise {type(e_gl).__name__} while get_node/get_child_nodes/prune return "
    f"None/[] for the same id — inconsistent contract; the 'rebuild the index' hint "
    f"is misleading for a genuinely-unknown id",
    "Low",
)

# get_node on a path that points at a FILE inside a LIVE node (known xfail).
# Own store/node so the result is deterministic (the loose file is still on disk).
fp_store = fresh(chunk=False)
with fp_store.create_node(step_type="m") as fpn:
    (fpn / "d.csv").write_text("x")
ok_fp, res_fp, e_fp = capture(fp_store.get_node, f"{fpn.node_id}/d.csv")
R.add(
    "5-API",
    "get_node on a path pointing at a FILE inside a live node",
    R.SURV if (ok_fp and res_fp is None) else R.DEGR,
    "returns None"
    if (ok_fp and res_fp is None)
    else f"raises {type(e_fp).__name__} instead of returning None — and the same gap "
    f"propagates through get_child_nodes / prune / from_parent (see the fuzzer)",
    "" if (ok_fp and res_fp is None) else "Low",
)

# operate on a store whose root dir was deleted out from under it
gone = fresh(chunk=True)
with gone.create_node(step_type="m") as n:
    n.add_meta("i", 1)
shutil.rmtree(gone.root)
expect(
    "query a store whose root dir was deleted mid-session", gone.find_node, (Exception,)
)  # any clean failure is safe; a hang would not be
expect(
    "create_node into a store whose root was deleted",
    lambda: _recreate(gone),
    (Exception, type(None)),
)

# -- Randomized fuzzer over every public method ------------------------------
fz = fresh(chunk=True, dedupe=True)
real = []
for i in range(8):
    with fz.create_node(
        step_type="m", parent=(real[-1] if real and i % 2 else None)
    ) as n:
        (n / "f.bin").write_text(str(i))
        n.add_meta("i", i)
    real.append(n)

methods = {
    "get_node": fz.get_node,
    "get_lineage": fz.get_lineage,
    "find_in_lineage": fz.find_in_lineage,
    "get_child_nodes": fz.get_child_nodes,
    "from_parent": lambda a: fz.from_parent(a, "*"),
    "prune_dry": lambda a: fz.prune(a, True),
    "get_most_recent_node": lambda a: fz.get_most_recent_node(step_type=a),
    "find_node": lambda a: fz.find_node(step_type=a),
}
arg_pool = [
    None,
    "",
    "   ",
    "none",
    "None",
    "deadbeef",
    "../../etc",
    "a/b/c",
    "🔥",
    "x" * 200,
    0,
    -1,
    3.14,
    [],
    {},
    real[0],
    real[-1].node_id,
    real[0].node_id + "/f.bin",
    float("nan"),
]
rng = random.Random(99)
# Acceptable = documented/graceful failure exception types. Any OTHER exception
# type (NotADirectoryError, RecursionError, AttributeError, ...) is a finding.
ACCEPTABLE = (ValueError, KeyError, TypeError, PermissionError)
unexpected = {}
calls = 0
for _ in range(600):
    name = rng.choice(list(methods))
    arg = rng.choice(arg_pool)
    calls += 1
    ok, res, e = capture(methods[name], arg)
    if not ok and not isinstance(e, ACCEPTABLE):
        unexpected.setdefault(f"{name}: {type(e).__name__}", []).append(repr(arg)[:30])
R.add(
    "5-API",
    f"randomized fuzzer: {calls} calls across {len(methods)} methods × "
    f"{len(arg_pool)} arg shapes",
    R.SURV if not unexpected else R.DEGR,
    "no undocumented exception types"
    if not unexpected
    else "undocumented exceptions: "
    + "; ".join(f"{k} (e.g. {v[0]})" for k, v in list(unexpected.items())[:6]),
    "" if not unexpected else "Low",
)


██████████████████████████████████████████████████████████████████████████████
█  CHAPTER 5 — ALL-METHODS ASSAULT & FUZZING
██████████████████████████████████████████████████████████████████████████████
  ✅ SURVIVED  resume before anything exists: get_most_recent_node on empty store
        └─ returned 'None'
  ✅ SURVIVED  prune a never-created id
        └─ returned '[]'
  ✅ SURVIVED  get_node on garbage id
        └─ returned 'None'
  ✅ SURVIVED  create with an illegal transition (rules violation)
        └─ failed safely: ValueError
  ✅ SURVIVED  create with a foreign/unknown parent id
        └─ failed safely: ValueError
  ✅ SURVIVED  prune the store ROOT directory
        └─ failed safely: PermissionError
  ✅ SURVIVED  from_parent on a root node (no parents)
        └─ returned '[]'
  ✅ SURVIVED  double context-exit / flush idempotency
        └─ returned 'None'
  ✅ SURVIVED  prune EVERYTHING then regenerate the web graph
        └─ returned '/var/folders/xf/n_m7ztrx4x577r3935n_1

  ⚠️ DEGRADED  randomized fuzzer: 600 calls across 8 methods × 19 arg shapes
        └─ undocumented exceptions: prune_dry: NotADirectoryError (e.g. '08dede96/f.bin'); get_child_nodes: NotADirectoryError (e.g. '08dede96/f.bin'); from_parent: NotADirectoryError (e.g. '08dede96/f.bin'); get_node: NotADirectoryError (e.g. '08dede96/f.bin')   [Low]


## Summary & findings

In [8]:
chapter("SUMMARY")
counts = R.counts()
print("Verdict tally:")
for v in (R.BROKE, R.DEGR, R.DESIGN, R.SURV, R.INFO):
    if counts.get(v):
        print(f"   {v}: {counts[v]}")
print(f"\nTotal probes: {len(R.rows)}\n")

# Group the notable findings (everything that isn't a clean SURVIVED/INFO)
print("NOTABLE FINDINGS (BROKE / DEGRADED / BY-DESIGN):")
print("-" * 78)
for r in R.rows:
    if r["verdict"] in (R.BROKE, R.DEGR, R.DESIGN):
        sev = f" [{r['severity']}]" if r["severity"] else ""
        print(f"{r['verdict']}{sev}  ({r['chapter']}) {r['title']}")
        if r["note"]:
            print(f"      {r['note']}")
print("-" * 78)
print(f"\nRSS now: {rss_mb():.0f} MB   |   work dir: {WORK}")
print("\n(Throwaway stores live under the work dir above — safe to delete.)")


██████████████████████████████████████████████████████████████████████████████
█  SUMMARY
██████████████████████████████████████████████████████████████████████████████
Verdict tally:
   💥 BROKE: 2
   ⚠️ DEGRADED: 13
   🔷 BY-DESIGN: 2
   ✅ SURVIVED: 30
   ℹ️ INFO: 3

Total probes: 50

NOTABLE FINDINGS (BROKE / DEGRADED / BY-DESIGN):
------------------------------------------------------------------------------
⚠️ DEGRADED [Low]  (2-IO) read artifact with a GARBAGE chunk (invalid zlib)
      raises raw error from zlib, not the friendly integrity error
⚠️ DEGRADED [Low]  (2-IO) read artifact with a MISSING chunk
      raises bare FileNotFoundError, not a clear 'chunk missing' error (known H8)
💥 BROKE [Medium]  (2-IO) one corrupt meta.json -> rebuild_db_from_disk() (the documented recovery)
      rebuild raises JSONDecodeError instead of skipping the bad node (confirms xfail 'test_rebuild_skips_corrupt_meta')
⚠️ DEGRADED [Low]  (2-IO) leftover meta.json.tmp after a crash mid-write
      